# Experiment: Kaggle Submission Exp 009 Raw 3fold

Objective:
- Run the first Kaggle submission for the raw `exp_009` noisy-student branch.
- Use the three native fold checkpoints as a plain probability ensemble.
- Avoid the old metadata priors layer because `exp_009b` showed it hurts this branch.

Success criteria:
- Generate `/kaggle/working/submission.csv` with the exact competition schema.
- Keep the notebook self-contained and offline-safe for Kaggle.


## Plan

- Load competition data and resolve the three `exp_009` fold checkpoints.
- Rebuild the native waveform classifier used in training.
- Discover hidden test soundscapes from `sample_submission.csv` stems.
- Run raw 3-fold probability ensembling on `5s` chunks.
- Save `submission.csv`.


In [ ]:
import os
import re
import time
import warnings
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings('ignore')
START = time.time()

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

if device.type == 'cpu':
    torch.set_num_threads(max(1, os.cpu_count() or 1))

if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

print('Torch:', torch.__version__)
print('Device:', device)
if device.type == 'cpu':
    print('CPU threads:', torch.get_num_threads())


In [ ]:
MODEL_DATASET_HINT = None  # Example: 'birdclef-exp009-raw-3fold'
CHECKPOINT_NAME = 'best_model.pt'
FOLD_IDS = (0, 1, 2)
AUDIO_SUFFIXES = {'.ogg', '.wav', '.flac', '.mp3'}


@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = 'slaney'
    mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    batch_chunks: int = 24

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)


cfg = Config()
INPUT_ROOT = Path('/kaggle/input')
row_pattern = re.compile(r'^(.*)_(\d+)$')


def resolve_project_root() -> Path | None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'birdclef-2026').exists():
            return candidate
    return None


PROJECT_ROOT = resolve_project_root()
if Path('/kaggle/working').exists():
    WORK_ROOT = Path('/kaggle/working')
elif PROJECT_ROOT is not None:
    WORK_ROOT = PROJECT_ROOT / 'submissions' / 'debug' / 'exp_009_raw_3fold'
else:
    WORK_ROOT = Path.cwd()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORK_ROOT / 'submission.csv'


def resolve_data_root() -> Path:
    direct_candidates = [
        INPUT_ROOT / 'competitions' / 'birdclef-2026',
        INPUT_ROOT / 'birdclef-2026',
    ]
    for candidate in direct_candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate

    if INPUT_ROOT.exists():
        for child in sorted(INPUT_ROOT.iterdir()):
            if (child / 'sample_submission.csv').exists():
                return child
            matches = sorted(child.rglob('sample_submission.csv'))
            if matches:
                return matches[0].parent

    if PROJECT_ROOT is not None:
        local_data = PROJECT_ROOT / 'data' / 'birdclef-2026'
        if (local_data / 'sample_submission.csv').exists():
            return local_data

    raise FileNotFoundError('BirdCLEF competition data was not found.')


def iter_candidate_roots(model_dataset_hint: str | None):
    if model_dataset_hint:
        hinted_path = Path(model_dataset_hint)
        if hinted_path.exists():
            yield hinted_path
        else:
            hinted_under_input = INPUT_ROOT / model_dataset_hint
            if hinted_under_input.exists():
                yield hinted_under_input

    if PROJECT_ROOT is not None:
        local_packaged = PROJECT_ROOT / 'submissions' / 'kaggle_datasets' / 'birdclef-exp009-raw-3fold'
        if local_packaged.exists():
            yield local_packaged
        local_outputs = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_009_noisy_student_pseudolabel'
        if local_outputs.exists():
            yield local_outputs

    if INPUT_ROOT.exists():
        for child in sorted(INPUT_ROOT.iterdir()):
            yield child


def resolve_fold_checkpoint_paths(model_dataset_hint: str | None = None, checkpoint_name: str = 'best_model.pt', fold_ids=FOLD_IDS) -> dict[int, Path]:
    searched = []
    seen = set()
    found = {}

    for root in iter_candidate_roots(model_dataset_hint):
        try:
            root = root.resolve()
        except FileNotFoundError:
            continue

        root_key = str(root)
        if root_key in seen:
            continue
        seen.add(root_key)
        searched.append(root_key)

        if root.is_dir():
            for path in sorted(root.rglob(checkpoint_name)):
                fold = None
                for part in path.parts:
                    if re.fullmatch(r'fold_\d{2}', part):
                        fold = int(part.split('_')[1])
                        break
                if fold in fold_ids and fold not in found:
                    found[int(fold)] = path
        elif root.is_file() and root.name == checkpoint_name:
            fold = None
            for part in root.parts:
                if re.fullmatch(r'fold_\d{2}', part):
                    fold = int(part.split('_')[1])
                    break
            if fold in fold_ids and fold not in found:
                found[int(fold)] = root

        if len(found) == len(tuple(fold_ids)):
            break

    missing = [int(fold) for fold in fold_ids if int(fold) not in found]
    if missing:
        raise FileNotFoundError(
            f'Could not find fold checkpoints for folds {missing}. Searched roots: {searched[:12]}'
        )
    return {int(fold): found[int(fold)] for fold in fold_ids}


def parse_row_id(row_id: str):
    match = row_pattern.match(str(row_id))
    if not match:
        return None, None
    return match.group(1), int(match.group(2))


def discover_test_files(data_root: Path, expected_stems: set[str], train_sc_dir: Path):
    search_roots = []
    for name in ['test_soundscapes', 'test_audio']:
        candidate = data_root / name
        if candidate.exists():
            search_roots.append(candidate)
    search_roots.append(data_root)

    matches = {}
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if not path.is_file() or path.suffix.lower() not in AUDIO_SUFFIXES:
                continue
            stem = path.stem
            if stem in expected_stems and stem not in matches:
                matches[stem] = path
        if len(matches) == len(expected_stems):
            break

    used_test_fallback = False
    if len(matches) == 0 and train_sc_dir.exists():
        used_test_fallback = True
        for stem in sorted(expected_stems):
            for suffix in sorted(AUDIO_SUFFIXES):
                candidate = train_sc_dir / f'{stem}{suffix}'
                if candidate.exists():
                    matches[stem] = candidate
                    break
        if matches:
            print('Warning: hidden test files not found; using aligned train fallback for a dry run.')

    ordered_paths = [matches[stem] for stem in sorted(matches)]
    return ordered_paths, used_test_fallback, sorted(expected_stems - set(matches))


DATA_ROOT = resolve_data_root()
TRAIN_SC_DIR = DATA_ROOT / 'train_soundscapes'
CHECKPOINT_PATHS = resolve_fold_checkpoint_paths(MODEL_DATASET_HINT, CHECKPOINT_NAME, FOLD_IDS)
SAMPLE_SUB = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
SPECIES = SAMPLE_SUB.columns[1:].tolist()
expected_ids = SAMPLE_SUB['row_id'].tolist()
expected_by_stem = {}
for row_id in expected_ids:
    stem, end_sec = parse_row_id(row_id)
    if stem is None:
        continue
    expected_by_stem.setdefault(stem, []).append(end_sec)
for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])
expected_stems = set(expected_by_stem)
test_files, used_test_fallback, missing_stems = discover_test_files(DATA_ROOT, expected_stems, TRAIN_SC_DIR)

print({
    'data_root': str(DATA_ROOT),
    'checkpoint_paths': {int(fold): str(path) for fold, path in CHECKPOINT_PATHS.items()},
    'num_classes': len(SPECIES),
    'matched_test_stems': len({path.stem for path in test_files}),
    'expected_test_stems': len(expected_stems),
    'missing_test_stems': len(missing_stems),
    'used_test_fallback': used_test_fallback,
    'submission_path': str(SUBMISSION_PATH),
    'device': str(device),
})
if test_files:
    print('First test path:', test_files[0])
if missing_stems:
    print('Missing test stems sample:', missing_stems[:5])


In [ ]:
@dataclass
class ReferenceModelConfig:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = 'slaney'
    mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0


class GEMFreqPool(nn.Module):
    def __init__(self, p_init: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        x = x.permute(0, 2, 1)
        att = torch.tanh(self.att_conv(x))
        att = F.softmax(att, dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        clipwise_prob = torch.sigmoid(clipwise_logit)
        return {
            'clipwise_logit': clipwise_logit,
            'clipwise_prob': clipwise_prob,
        }


class SEDModel(nn.Module):
    def __init__(self, cfg: ReferenceModelConfig):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=False,
            in_chans=cfg.in_channels,
            features_only=False,
            global_pool='',
            num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(feat_dim, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.gem_pool(features)
        return self.head(pooled)


class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg: ReferenceModelConfig):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.fmin,
            f_max=cfg.fmax,
            power=cfg.power,
            norm=cfg.norm,
            mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.mel(waveforms)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.db(mel)
        mel = torch.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)

        batch_size = mel.shape[0]
        mel_flat = mel.reshape(batch_size, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)


class WaveformSEDClassifier(nn.Module):
    def __init__(self, cfg: ReferenceModelConfig):
        super().__init__()
        self.mel = MelSpectrogramTransform(cfg)
        self.model = SEDModel(cfg)

    def forward(self, waveforms):
        mel = self.mel(waveforms)
        return self.model(mel)


def safe_load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        return ckpt['model_state_dict']
    return ckpt


def extract_checkpoint_meta(ckpt):
    if not isinstance(ckpt, dict):
        return {'stage': 'raw_state_dict'}
    metrics = ckpt.get('metrics', {})
    return {
        'epoch': ckpt.get('epoch'),
        'resume_mode': ckpt.get('resume_mode'),
        'metrics': metrics if isinstance(metrics, dict) else {},
    }


def load_model(ckpt_path: Path, cfg_native: Config):
    model_cfg = ReferenceModelConfig(num_classes=cfg_native.num_classes)
    model = WaveformSEDClassifier(model_cfg)
    ckpt = safe_load_checkpoint(ckpt_path)
    model.load_state_dict(extract_state_dict(ckpt), strict=True)
    model.to(device).eval()
    return model, extract_checkpoint_meta(ckpt)


ensemble = []
checkpoint_meta = {}
for fold in FOLD_IDS:
    model, meta = load_model(CHECKPOINT_PATHS[int(fold)], cfg)
    ensemble.append((int(fold), model))
    checkpoint_meta[int(fold)] = meta

print('Loaded folds:', [int(fold) for fold, _ in ensemble])
print(checkpoint_meta)


In [ ]:
def load_soundscape(path: Path) -> np.ndarray:
    audio, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def predict_chunk_probs(model, chunk_tensor: torch.Tensor, batch_size: int) -> np.ndarray:
    outputs = []
    amp_enabled = device.type == 'cuda'
    for start in range(0, chunk_tensor.size(0), batch_size):
        batch = chunk_tensor[start: start + batch_size]
        with (torch.amp.autocast(device_type='cuda', enabled=amp_enabled) if amp_enabled else nullcontext()):
            out = model(batch)
        outputs.append(out['clipwise_prob'].detach().float().cpu().numpy().astype(np.float32))
    return np.concatenate(outputs, axis=0)


def build_submission(row_ids, preds, expected_ids, species):
    if len(preds) == 0:
        pred_df = pd.DataFrame(
            np.zeros((0, len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index([], name='row_id'),
        )
    else:
        pred_df = pd.DataFrame(
            np.asarray(preds, dtype=np.float32),
            columns=species,
            index=pd.Index(row_ids, name='row_id'),
        )

    pred_df = pred_df[~pred_df.index.duplicated(keep='first')]

    missing_ids = [row_id for row_id in expected_ids if row_id not in pred_df.index]
    if missing_ids:
        zeros = pd.DataFrame(
            np.zeros((len(missing_ids), len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index(missing_ids, name='row_id'),
        )
        pred_df = pd.concat([pred_df, zeros], axis=0)

    pred_df = pred_df.loc[expected_ids]
    return pred_df.reset_index(), len(missing_ids)


In [ ]:
CHUNK = cfg.chunk_samples
all_row_ids = []
all_preds = []

print('Running exp_009 raw 3-fold ensemble inference...')
with torch.no_grad():
    for file_idx, path in enumerate(test_files, start=1):
        stem = path.stem
        audio = load_soundscape(path)
        expected_secs = expected_by_stem.get(stem, [])

        if expected_secs:
            n_chunks = max(1, max(expected_secs) // int(cfg.chunk_duration))
        else:
            n_chunks = max(1, int(np.ceil(len(audio) / CHUNK)))

        padded_len = n_chunks * CHUNK
        if len(audio) < padded_len:
            audio = np.pad(audio, (0, padded_len - len(audio)))
        else:
            audio = audio[:padded_len]

        chunks = audio.reshape(n_chunks, CHUNK)
        chunk_tensor = torch.from_numpy(chunks).float().to(device)

        if expected_secs:
            valid_indices = [(end_sec // int(cfg.chunk_duration)) - 1 for end_sec in expected_secs]
            valid_indices = [idx for idx in valid_indices if 0 <= idx < n_chunks]
            if not valid_indices:
                continue
            target_row_ids = [f'{stem}_{(idx + 1) * int(cfg.chunk_duration)}' for idx in valid_indices]
        else:
            valid_indices = list(range(n_chunks))
            target_row_ids = [f'{stem}_{(idx + 1) * int(cfg.chunk_duration)}' for idx in valid_indices]

        fold_probs = []
        for fold, model in ensemble:
            probs = predict_chunk_probs(model, chunk_tensor, cfg.batch_chunks)
            fold_probs.append(probs[valid_indices])

        ensemble_probs = np.mean(np.stack(fold_probs, axis=0), axis=0)
        all_row_ids.extend(target_row_ids)
        all_preds.extend(ensemble_probs)

        if file_idx % 10 == 0 or file_idx == len(test_files):
            print(f'Processed {file_idx}/{len(test_files)} files')

submission, missing_count = build_submission(all_row_ids, all_preds, expected_ids, SPECIES)
submission.to_csv(SUBMISSION_PATH, index=False)

print('Missing row_ids filled with zeros:', missing_count)
print('Submission shape:', submission.shape)
print('Saved:', SUBMISSION_PATH)
print('Elapsed minutes:', round((time.time() - START) / 60.0, 2))
submission.head()
